# 09p The personal map against a learned low-rank adapter (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. A shared network with a SuLoRA output layer (Klein et al., arXiv 2510.08059: a shared read-out plus a rank-8
adapter per donor) is trained on the nine donors. For the new person: (a) a new adapter fitted on their k labelled
images with everything shared frozen; (b) the donor adapter that fits those images best, copied and fine-tuned (the
calibration route of Lopes et al., arXiv 2606.16462); (c) the shared part alone. Weight decay by 5-fold
cross-validation on the k images. Against borrowing and the personal map at k = 50, 100, 250, 500.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09p-adapter', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [50, 100, 250, 500], "primary_ks": [50, 100, 250], "draws": [13, 14, 15], "rank": 8, "epochs": 30, "lr": 0.001, "batch": 256, "hold_images": 1000, "adapter_steps": 300, "adapter_lr": 0.01, "adapter_wds": [0.0001, 0.01, 1.0], "window_s": [0.1, 0.2], "seed": 342}')
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']
TAG = '09p'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models import sulora as su
from cross_subject_encoding.utils import config as C

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
n_ch, n_t = 17, len(times)
N_TRAIN = len(next(iter(y_train.values())))
win = ha.window_index(times, 100)
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
post = np.repeat((times >= -1e-9)[None, :], n_ch, 0).ravel()

def sources(ztr, zte):
    tr = np.empty((len(people), len(ztr), n_ch * n_t), np.float32)
    te_ = np.empty((len(people), len(zte), n_ch * n_t), np.float32)
    q = np.concatenate([ztr, zte])
    for i, s in enumerate(people):
        pred = pl.ols_paper(ztr, y_train[s].reshape(len(ztr), -1), q)
        tr[i], te_[i] = pred[:len(ztr)], pred[len(ztr):]
    return tr, te_

def similarity(pred, meas):
    # centred over the 200 test images, then Pearson over the post-onset features
    p, m = pred[:, post] - pred[:, post].mean(0), meas[:, post] - meas[:, post].mean(0)
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]  # (measured, candidates)

def calibrate(r, how):
    if how == "none":
        return r
    if how == "csls":  # cross-domain similarity local scaling (Conneau et al. 2018), k = 10, as SATTC's geometric expert
        rm = np.sort(r, 1)[:, -10:].mean(1)
        rp = np.sort(r, 0)[-10:, :].mean(0)
        return 2 * r - rm[:, None] - rp[None, :]
    if how == "dual_z":  # each candidate's similarities standardized over the test responses, then each response's
        z = (r - r.mean(0)) / (r.std(0) + 1e-30)
        return (z - z.mean(1, keepdims=True)) / (z.std(1, keepdims=True) + 1e-30)
    raise ValueError(how)

def test_time_map(pred, meas):
    # label-free, transductive (the 200 test responses as a batch, no pairing used): per 100 ms window, re-mix the
    # prediction's channels so that their covariance over images and time points matches the measured responses'
    from scipy.linalg import sqrtm
    p3, m3 = t3(pred, len(pred)), t3(meas, len(meas))
    out = p3.copy()
    for w in np.unique(win):
        cols = win == w
        xp = (p3[:, :, cols] - p3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        xm = (m3[:, :, cols] - m3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        cp = np.cov(xp, rowvar=False) + 1e-6 * np.eye(n_ch)
        cm = np.cov(xm, rowvar=False) + 1e-6 * np.eye(n_ch)
        a = np.real(np.linalg.inv(sqrtm(cp)) @ sqrtm(cm))
        mean = p3[:, :, cols].mean(0)
        out[:, :, cols] = np.einsum("nct,cd->ndt", p3[:, :, cols] - mean, a) + mean
    return out.reshape(len(pred), -1)

def encoding_r(pred, meas):
    # per-output correlation across the 200 test images with the repetition average, 100-200 ms, averaged
    sel = np.repeat(((times >= 0.1) & (times < 0.2))[None, :], n_ch, 0).ravel()
    p, m = pred[:, sel] - pred[:, sel].mean(0), meas[:, sel] - meas[:, sel].mean(0)
    return float(np.mean((p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)))

def accuracy(r):
    rank = (r > np.diag(r)[:, None]).sum(1)
    return float((rank < 1).mean() * 100), float((rank < 5).mean() * 100)

def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x))}

import torch
torch.set_num_threads(os.cpu_count() or 4)
N_TEST = len(z_test)
src_train, src_test = sources(z_train, z_test)
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
xt = torch.as_tensor(z_train, dtype=torch.float32)
xq = torch.as_tensor(z_test, dtype=torch.float32)
hold = np.random.default_rng(FIXED["seed"]).choice(N_TRAIN, FIXED["hold_images"], replace=False)
train_idx = np.setdiff1d(np.arange(N_TRAIN), hold)
rows, dec_rows, train_log = [], [], []
for i, person in enumerate(people):
    t = time.monotonic()
    others = [people[j] for j in range(len(people)) if j != i]
    # outputs standardized with the donors' pooled mean and spread (no target data)
    pooled = np.stack([y_train[s].reshape(N_TRAIN, -1) for s in others])
    mu, sd = pooled.mean((0, 1)), pooled.std((0, 1)) + 1e-6
    del pooled
    ys = {s: torch.as_tensor((y_train[s].reshape(N_TRAIN, -1) - mu) / sd, dtype=torch.float32) for s in others}
    model = su.SuLoRAEncoder(xt.shape[1], ys[others[0]].shape[1], others, rank=FIXED["rank"], seed=FIXED["seed"] + i)
    fit = su.train_donors(model, xt, ys, train_idx, hold, FIXED["epochs"], FIXED["lr"], FIXED["batch"], FIXED["seed"] + i, log=lambda m: None)
    train_log.append({"person": person, **fit})
    del ys
    unstd = lambda p: (np.asarray(p) * sd + mu).reshape(N_TEST, n_ch, n_t)
    others_i = [j for j in range(len(people)) if j != i]
    avg_train, avg_t3 = src_train[others_i].mean(0), t3(src_test[others_i].mean(0), N_TEST)
    preds = {"avg|0|-": avg_t3, "shared|0|-": unstd(su.adapt_new_person(model, xt[:1], torch.zeros(1, n_ch * n_t), xq, "shared"))}
    for d in FIXED["draws"]:
        for k in FIXED["ks"]:
            idx = order(person, d, k)
            y_k = np.asarray(y_train[person][idx], np.float64)
            preds[f"h4b|{k}|{d}"] = h4b.apply_map(h4b.fit_map(t3(avg_train[idx], k), y_k, win, seed=d)[0], avg_t3, win)
            yk_std = torch.as_tensor((y_k.reshape(k, -1) - mu) / sd, dtype=torch.float32)
            for route in ("new", "best_donor"):
                preds[f"{route}|{k}|{d}"] = unstd(su.adapt_new_person(model, xt[idx], yk_std, xq, route, wds=tuple(FIXED["adapter_wds"]),
                                                                       steps=FIXED["adapter_steps"], lr=FIXED["adapter_lr"], seed=d))
    preds = {name: np.asarray(p, np.float64).reshape(N_TEST, n_ch, n_t) for name, p in preds.items()}
    if not all(np.isfinite(p).all() for p in preds.values()):
        raise ValueError(f"{person}: nonfinite prediction")
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    meas = bio_test[person].mean(1).reshape(N_TEST, -1).astype(np.float64)
    for name, curve in scored["correlation"].items():
        method, k, d = name.split("|")
        rows.append({"person": person, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean()),
                     "top1_centred": accuracy(similarity(preds[name].reshape(N_TEST, -1), meas))[0]})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    pd.DataFrame(train_log).to_csv(C.METRICS_DIR / f"{TAG}_training.csv", index=False)
    print(person, f"best epoch {fit['best_epoch']} (held-out r {fit['best_score']:.3f}) | {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(method, k, metric="r_100_200ms"):
    f = frame[(frame.method == method) & (frame.k == str(k))]
    return f.groupby("person")[metric].mean().reindex(people).to_numpy()
summary = {"post_hoc": "development; adapted comparator (SuLoRA, Klein et al.; calibration route, Lopes et al.)",
           "k0": {m: {"r": float(per_person(m, 0).mean()), "top1": float(per_person(m, 0, "top1_centred").mean())} for m in ("avg", "shared")},
           "by_k": {}}
for k in FIXED["ks"]:
    block = {}
    for m in ("h4b", "new", "best_donor"):
        block[m] = {"r": float(per_person(m, k).mean()), "top1": float(per_person(m, k, "top1_centred").mean())}
    for m in ("new", "best_donor"):
        block[f"h4b_minus_{m}_r"] = ci(per_person("h4b", k) - per_person(m, k))
        block[f"h4b_minus_{m}_top1"] = ci(per_person("h4b", k, "top1_centred") - per_person(m, k, "top1_centred"))
    summary["by_k"][k] = block
pk = FIXED["primary_ks"]
best_adapter = np.maximum(np.mean([per_person("new", k) for k in pk], 0), np.mean([per_person("best_donor", k) for k in pk], 0))
summary["primary_h4b_minus_best_adapter_r_k50_250"] = ci(np.mean([per_person("h4b", k) for k in pk], 0) - best_adapter)
summary["rule"] = ("if an adapter route beats H4b at k <= 250 (primary contrast's upper bound below 0), the paper's claim narrows to "
                   "'a linear map is competitive with a learned adapter'; reported either way")
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ks = FIXED["ks"]
for a, metric, label in zip(ax, ("r", "top1"), ("encoding r, 100-200 ms", "top-1 of 200 (%), centred")):
    a.axhline(summary["k0"]["avg"][metric], color="#7A7A7A", ls="--", label="others' average")
    a.axhline(summary["k0"]["shared"][metric], color="#9DB4D9", ls=":", label="SuLoRA shared part (k = 0)")
    for m, colour, name in (("h4b", "#C8102E", "personal map (H4b)"), ("new", "#2E5EAA", "SuLoRA, new adapter"),
                            ("best_donor", "#E8A33D", "best donor adapter, fine-tuned")):
        a.plot(range(len(ks)), [summary["by_k"][k][m][metric] for k in ks], "o-", color=colour, label=name)
    a.set(xticks=range(len(ks)), xticklabels=[str(k) for k in ks], xlabel="labelled images k", ylabel=label)
ax[0].legend(fontsize=7)
ax[0].set_title("09p: map vs a learned low-rank adapter")
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({k: {m: (round(v["r"], 4), round(v["top1"], 1)) for m, v in b.items() if isinstance(v, dict) and "r" in v} for k, b in summary["by_k"].items()}, indent=1))
print(summary["primary_h4b_minus_best_adapter_r_k50_250"])